# pandas を利用したデータ操作

- Python の pandas というライブラリを利用すると，dataframe という形式の表データを取り扱うことができる
- pandas を利用するには，まず `import pandas as pd` を実行する

※ 各コードの意味は理解しなくてもよい．実行の前後のデータから，何をしているのかをよく見ておくこと

In [ ]:
# pandas ライブラリをインポートする (使える状態にする)
import pandas as pd

# 完了確認のメッセージ
print("完了!")

## dataframe の操作

- `pd.read_excel` という関数を用いると，**Excel ファイルの読み込み**ができる
- ほかには，CSV ファイルを読み込む `pd.read_csv` という関数もある

In [ ]:
# data08.xlsx という Excel ファイルの「科目」というワークシートを読み込み，
# dataframe という形式で変数 subjects に代入する
subjects = pd.read_excel("data08.xlsx", sheet_name="科目")

# 変数 subjects の内容の表示
display(subjects)
# 変数の内容は，上のツールバーの「変数」からも確認できる

In [ ]:
# data08.xlsx の「クラス」を classes に読み込み，内容を表示する
classes = pd.read_excel("data08.xlsx", sheet_name="クラス")
display(classes)

In [ ]:
# data08.xlsx の「専任教員」を fulltime に読み込み，内容を表示する
fulltime = pd.read_excel("data08.xlsx", sheet_name="専任教員")
display(fulltime)

In [ ]:
# data08.xlsx の「非常勤教員」を parttime に読み込み，内容を表示する
parttime = pd.read_excel("data08.xlsx", sheet_name="非常勤教員")
display(parttime)

- まず，fulltime と parttime を1つの表 teachers にまとめる（**和**; **union**）
- 2つの表 (dataframe) の和は，`pd.concat` 関数で行える

※ 厳密には，さらに**重複した行を1行にまとめる**必要がある（その場合は，`.drop_duplicates()` を付ける）

In [ ]:
# fulltime と parttime の和を teachers に代入し，内容を表示する
teachers = pd.concat([fulltime, parttime], ignore_index=True)   # .drop_duplicates()
display(teachers)

- 3つの表 (subjects, classes, teachers; いずれも dataframe) を**結合 (join)** する
- 2つの表 (dataframe) の結合は，`pd.merge` 関数で行える

In [ ]:
# まず，classes と subjects を結合し，
# 結果の表 (dataframe) を joined1 に代入し，内容を表示する
joined1 = pd.merge(classes, subjects, on="科目番号")
display(joined1)

In [ ]:
# 次に，先に結合した joined1 と teachers を結合し，
# 結果の表 (dataframe) を joined2 に代入し，内容を表示する
joined2 = pd.merge(joined1, teachers, on="教員ID")
display(joined2)

- dataframe で**選択** (**selection**; 特定の行を選び出す操作) を行う方法はいくつかある
- ここでは，"期" が "後" である行だけを選ぶ

In [ ]:
# joined2 から，"期" が "後" に等しいという条件を満たす行を選択し，
# 結果の表 (dataframe) を selected に代入し，内容を表示する
selected = joined2[joined2["期"]=="後"]
display(selected)

- **射影** (**projection**; 特定の列を選び出す操作) を行う方法もいくつかある
- ここでは，"科目番号", "科目名", "クラス", "教員名", "曜/時", "教室"
  をこの順に並べる

※ 厳密な意味での射影では，さらに**重複した行を1行にまとめる**必要がある（その場合は，`.drop_duplicates()` を付ける）

In [ ]:
# selected から，指定した列を指定した順に抜き出して，
# 結果の表 (dataframe) を projected に代入し，内容を表示する
projected = selected[["科目番号", "科目名", "クラス", "教員名", "曜/時", "教室"]]   # .drop_duplicates()
display(projected)

- 行の**並べ替え**は，`.sort_values` というメソッド (データの直後に付けて，データの操作をする関数) で行える

In [ ]:
# projected の各行を，"科目番号" の順に並べ替える
# "科目番号" の等しいものに関しては，"クラス" の順に並べ替える
sorted = projected.sort_values(["科目番号", "クラス"])
display(sorted)

- `.reset_index` メソッドで**インデックス (通し番号) の振り直し**ができる

In [ ]:
# sorted のインデックスを振り直して，
# 結果の表 (dataframe) を final に代入する
final = sorted.reset_index(drop=True)
display(final)

- `.to_csv` メソッドで，**CSVファイルへの保存**ができる

In [ ]:
# 出来上がった表 (final) を，CSVファイル (data08final.csv) に保存する
# encoding="utf_8_sig" は，文字化け対策のために必要
final.to_csv("data08final.csv", index=False, encoding="utf_8_sig")

# 完了確認のメッセージ
print("完了!")

In [ ]:
# 確認のため，改めて data08final.csv 読み込んで (dataframe に変換して) 表示する
reloaded = pd.read_csv("data08final.csv")
display(reloaded)

## データクレンジング
データに表記のゆらぎや欠損値がある場合，データ処理が正しく行えないことがあるため，あらかじめ対処しておくこと．

- 表記のゆらぎがある場合，表記を統一する
- 欠損値 (NaN) のあるデータについては
  - 欠損値を適切な値で埋める
    - 同じレコードの他の値から求まる場合は，その値で埋める
    - 同じ列のデータを用いて，代表値 (平均値，中央値，最頻値など) で埋める

    など
  - 欠損値を含むレコード (行) を削除する

  のいずれかを行う

※ 各コードの意味は理解しなくてもよい．実行の前後のデータから，何をしているのかをよく見ておくこと

In [ ]:
import pandas as pd

# サンプルデータの読み込みと表示
# data08.xlsx の「クレンジング前」を uriage_data に読み込み，内容を表示する
# (header=2 は，最初の2行を読み飛ばすためのもの)
uriage_data = pd.read_excel("data08.xlsx", sheet_name="クレンジング前", header=2)
display(uriage_data)

### 表記のゆらぎの統一

In [ ]:
# 表記のゆらぎの統一 (商品名の "はさみ", "ハサミ" は "ハサミ" に統一)
uriage_data["商品名"] = uriage_data["商品名"].str.replace("はさみ", "ハサミ")
display(uriage_data)

### 欠損値の補充

In [ ]:
# 曜日の欠損値を日付から埋める
wd = ('月','火','水','木','金','土','日')
nans = uriage_data["曜日"].isna()
for i, row in uriage_data.loc[nans,:].iterrows():
    uriage_data.loc[i, "曜日"] = wd[row["日付"].weekday()]
display(uriage_data)


In [ ]:
# 顧客の性別の欠損値を最頻値で埋める
mode_of_gender = uriage_data["顧客の性別"].mode()[0]   # 顧客の性別の最頻値
uriage_data["顧客の性別"] = uriage_data["顧客の性別"].fillna(mode_of_gender)
display(uriage_data)

In [ ]:
# 顧客の年齢の欠損値を平均値（を四捨五入した値）で埋める
mean_of_age = round(uriage_data["顧客の年齢"].mean())   # 顧客の年齢の平均値
uriage_data["顧客の年齢"] = uriage_data["顧客の年齢"].fillna(mean_of_age)
display(uriage_data)

### 保存

In [ ]:
# 出来上がった表 (uriage_data) を，CSVファイル (uriage_data.csv) に保存する
# encoding="utf_8_sig" は，文字化け対策のために必要
uriage_data.to_csv("uriage_data.csv", index=False, encoding="utf_8_sig")

# 完了確認のメッセージ
print("完了!")